# Capacity Planning

This notebook translates benchmark results into actionable team capacity recommendations — how many developers a deployment can support, when to add replicas, and what it costs.

**What we'll do:**
1. Define workload assumptions
2. Calculate single-replica developer capacity
3. Project multi-replica scaling
4. Analyze cost per developer
5. Generate recommendations by team size

## 1. Workload Assumptions

These defaults represent a typical coding assistant workload. Adjust based on your team's usage patterns.

In [ ]:
SINGLE_USER_TOKS = 93       # Output tok/s from synchronous benchmark (L40S reference)
PEAK_AGG_TOKS = 1357        # Peak aggregate tok/s from throughput benchmark
TTFT_SLO_MS = 500           # Maximum acceptable TTFT for interactive coding
CONCURRENCY_RATIO = 0.30    # Fraction of active devs hitting model simultaneously
AVG_PROMPT_TOKENS = 512     # System prompt + context + user message
AVG_OUTPUT_TOKENS = 256     # Typical code generation response
GPU_COST_PER_HOUR = 2.50    # $/hr for GPU instance (L40S g6e.2xlarge ≈ $2.24)

print("=== Workload Assumptions ===")
print(f"  Single-user output tok/s:  {SINGLE_USER_TOKS}")
print(f"  Peak aggregate tok/s:      {PEAK_AGG_TOKS}")
print(f"  TTFT SLO target:           {TTFT_SLO_MS} ms")
print(f"  Concurrency ratio:         {CONCURRENCY_RATIO:.0%}")
print(f"  Avg prompt tokens:         {AVG_PROMPT_TOKENS}")
print(f"  Avg output tokens:         {AVG_OUTPUT_TOKENS}")
print(f"  GPU cost/hr:               ${GPU_COST_PER_HOUR:.2f}")
print(f"\n💡 Update these values with your benchmark results from the previous notebook.")

## 2. Single-Replica Capacity

Calculate how many developers one GPU replica can support while maintaining the TTFT SLO.

In [ ]:
theoretical_max = PEAK_AGG_TOKS / (SINGLE_USER_TOKS * CONCURRENCY_RATIO)
practical_capacity = int(theoretical_max * 0.6)  # 60% headroom for SLO compliance

print("=== Single-Replica Capacity ===")
print(f"  Theoretical maximum:  {theoretical_max:.0f} developers")
print(f"  With SLO headroom:    {practical_capacity} developers")
print(f"")
print(f"  Formula: peak_agg_toks / (single_user_toks × concurrency_ratio) × 0.6")
print(f"           {PEAK_AGG_TOKS} / ({SINGLE_USER_TOKS} × {CONCURRENCY_RATIO}) × 0.6 = {practical_capacity}")
print(f"")
print(f"  At {practical_capacity} developers:")
print(f"    Concurrent requests: ~{int(practical_capacity * CONCURRENCY_RATIO)}")
print(f"    Per-user tok/s:      ~{PEAK_AGG_TOKS / max(1, practical_capacity * CONCURRENCY_RATIO):.0f}")

## 3. Multi-Replica Scaling

llm-d EPP routes requests across replicas with prefix-cache affinity. Throughput scales near-linearly with replicas (assume 90% efficiency due to routing overhead).

In [ ]:
ROUTING_EFFICIENCY = 0.90

print("=== Multi-Replica Scaling ===\n")
print(f"{'Replicas':<10} {'Agg tok/s':<15} {'Dev Capacity':<15} {'Monthly Cost':<15}")
print("-" * 55)

for replicas in [1, 2, 3, 4]:
    scaled_toks = PEAK_AGG_TOKS * replicas * ROUTING_EFFICIENCY
    capacity = int((scaled_toks / (SINGLE_USER_TOKS * CONCURRENCY_RATIO)) * 0.6)
    monthly_cost = GPU_COST_PER_HOUR * 8 * 22 * replicas  # 8hr/day, 22 days/month
    print(f"{replicas:<10} {scaled_toks:<15.0f} {capacity:<15} ${monthly_cost:<14.0f}")

print(f"\n  Efficiency factor: {ROUTING_EFFICIENCY:.0%} (accounts for EPP routing overhead)")
print(f"  Cost assumes 8hr/day operation (scale to zero overnight)")

## 4. Cost Analysis

Compare cost per developer across different GPU tiers and team sizes.

In [ ]:
print("=== Cost Per Developer Per Month ===\n")
print("(Assumes 8hr/day, 22 working days, single replica)\n")

gpu_options = [
    ("L10 (24GB)", 1.50, 500, 50),
    ("L40S (48GB)", 2.50, 1357, 93),
    ("A100 (80GB)", 4.00, 2781, 138),
]

print(f"{'GPU':<15} {'$/hr':<8} {'Peak tok/s':<12} {'Capacity':<12} {'$/dev/month':<12} {'$/1M tokens':<12}")
print("-" * 71)

for name, cost_hr, peak_toks, user_toks in gpu_options:
    capacity = int((peak_toks / (user_toks * CONCURRENCY_RATIO)) * 0.6)
    monthly = (cost_hr * 8 * 22) / max(1, capacity)
    cost_per_m_tokens = (cost_hr * 3600) / peak_toks / 1_000_000 * 1_000_000
    print(f"{name:<15} ${cost_hr:<7.2f} {peak_toks:<12} {capacity:<12} ${monthly:<11.0f} ${cost_per_m_tokens:<11.2f}")

## 5. Recommendations by Team Size

In [ ]:
print("=== Deployment Recommendations ===\n")

team_sizes = [5, 10, 15, 20, 30, 50]

print(f"{'Team Size':<12} {'GPU Recommendation':<30} {'Replicas':<10} {'Est. $/month':<12}")
print("-" * 64)

for team in team_sizes:
    if team <= 8:
        gpu, cost, cap = "L10 (24GB) — entry tier", 1.50, 6
    elif team <= 15:
        gpu, cost, cap = "L40S (48GB) — sweet spot", 2.50, practical_capacity
    elif team <= 30:
        gpu, cost, cap = "L40S (48GB)", 2.50, practical_capacity
    else:
        gpu, cost, cap = "A100 (80GB) or 2×L40S", 4.00, 29

    replicas = max(1, -(-team // cap))  # ceiling division
    monthly = cost * 8 * 22 * replicas
    print(f"{team:<12} {gpu:<30} {replicas:<10} ${monthly:.0f}")

print(f"\n💡 These are estimates. Run Phase 6 benchmarks on YOUR deployment for accurate numbers.")
print(f"   Actual capacity depends on prompt size, cache hit rate, and concurrency patterns.")

## Summary

| Metric | Value |
|--------|-------|
| Single-replica capacity | ~{practical_capacity} developers (with SLO headroom) |
| Scaling efficiency | 90% per additional replica |
| Best cost/dev | A100 at scale (~$27-40/dev/month) |
| Best entry point | L10 for ≤8 devs (~$60-96/dev/month) |
| Sweet spot | L40S for 10-15 dev teams (~$40-60/dev/month) |

**Key takeaways:**
- Larger GPUs have lower cost-per-developer despite higher hourly rates
- Prefix caching (llm-d) improves effective capacity by reducing redundant prefill
- Scale replicas horizontally — llm-d EPP maintains prefix-cache benefits across replicas
- Budget for 8hr/day operation — scale GPU nodes to zero overnight for ~67% cost savings

→ Continue to **Phase 7** (`7_advanced/`) for multi-accelerator and multi-cloud deployment patterns.